# 🤖 Introduzione all'AI Agentica
## Cosa sono gli Agenti AI & Primi Passi con Tool Calling (Ollama + LangChain)

---

### 🎯 Obiettivo del Notebook
In questo notebook esploreremo i concetti fondamentali dei **sistemi agentici**:
1. **Workflow vs Agente**: qual è la differenza chiave?
2. **Il ciclo di un Agente (Reasoning Loop)**: come l'LLM decide quali azioni compiere.
3. **Tool Calling con Ollama**: come definire funzioni Python (ad es. *scrivere un file*, *leggere file*, *fare calcoli*) e metterle a disposizione del modello.
4. **Agent Loop Manuale (ReAct da zero)**: per capire esattamente cosa accade «sotto il cofano» quando un modello invoca un tool e ne riceve il risultato.
5. **Interazione Multi-Turno con l'Utente**: continuare la conversazione dopo l'esecuzione del tool e visualizzare l'intera cronologia dei messaggi.
6. **Creare un Agente autonomo completo**: interazione conversazionale con memoria e tool multipli.

| Sezione | Argomento | Descrizione |
|---|---|---|
| **1. Teoria** | Concetti Chiave | Workflow vs Agenti, anatomia di un Agent Loop |
| **2. Setup** | Configurazione | Modello Ollama e import delle librerie |
| **3. Tool Calling Base** | Il nostro primo Tool | Creare il tool `scrivi_file` e bindarlo a Ollama |
| **4. Esecuzione del Tool & Cronologia** | Tool Execution + User Reply | Esecuzione tool, risposta LLM, feedback utente e visualizzazione completa |
| **5. Agente Multi-Tool** | Ecosistema di Tool | Scrittura file, lettura file e calcolatrice controllati dall'agente |
| **6. Mini-Progetto** | Task Autonomo | L'agente analizza una richiesta, genera un file ed effettua verifiche |

> **Requisiti:** Ollama in esecuzione su `localhost:11434` (ad es. modello `llama3.2` o `qwen2.5`).

---
## 📚 1. Workflow vs Agenti: Qual è la differenza?

Nel notebook `01_workflow_design_patterns.ipynb` abbiamo visto i **Workflow**:
- In un **Workflow**, i passaggi sono definiti a priori dallo sviluppatore nel codice (es. Chaining, Parallelism, Routing predeterminato).
- In un **Agente AI**, l'LLM ha a disposizione un insieme di **Tool** (strumenti) e decide **in autonomia ad ogni ciclo**:
  1. Se ha bisogno di usare un tool o può rispondere direttamente.
  2. Quale tool chiamare e con quali argomenti.
  3. Come interpretare il risultato del tool (Observation) per decidere il passo successivo (**Reasoning Loop**).

---
## ⚙️ 2. Setup — Installazione e Configurazione

In [ ]:
# Installa le librerie necessarie se non già presenti nell'ambiente
%pip install -q langchain langchain-ollama langchain-core

In [1]:
import os
import json
from typing import Any, List, Dict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool

# Configurazione Ollama
MODEL_NAME = "qwen2.5:latest  "  # Modello raccomandato per function calling locale (o 'qwen2.5')
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)

# Test rapido di connessione ad Ollama
try:
    test_response = llm.invoke("Rispondi solo con: 'Ollama è pronto!'")
    print(f"✅ Connessione riuscita: {test_response.content.strip()}")
except Exception as e:
    print(f"❌ Errore di connessione a Ollama: {e}")
    print("Verifica che Ollama sia avviato su terminale con 'ollama serve' e che il modello sia presente.")

✅ Connessione riuscita: Ollama è pronto!


### 🎨 Utility: Visualizzare la conversazione completa in modo chiaro
Definiamo una funzione `mostra_conversazione(messaggi)` per visualizzare tutti i tipi di messaggio che compongono lo storico: `SystemMessage`, `HumanMessage`, `AIMessage` (anche con Tool Call) e `ToolMessage`.

In [2]:
def mostra_conversazione(messaggi: List[BaseMessage]):
    """
    Stampa l'intera cronologia dei messaggi formattata con icone e ruoli.
    """
    print("=" * 70)
    print("📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE")
    print("=" * 70)
    
    for i, msg in enumerate(messaggi, 1):
        if isinstance(msg, SystemMessage):
            print(f"\n[{i}] ⚙️ SYSTEM:")
            print(f"    {msg.content}")
            
        elif isinstance(msg, HumanMessage):
            print(f"\n[{i}] 👤 UTENTE:")
            print(f"    {msg.content}")
            
        elif isinstance(msg, AIMessage):
            print(f"\n[{i}] 🤖 ASSISTENTE (AI):")
            if msg.content:
                print(f"    {msg.content}")
            if getattr(msg, 'tool_calls', None):
                for tc in msg.tool_calls:
                    print(f"    🛠️ [Tool Call] Funzione: '{tc['name']}' | Parametri: {json.dumps(tc['args'], ensure_ascii=False)}")
                    
        elif isinstance(msg, ToolMessage):
            print(f"\n[{i}] 📥 RISULTATO TOOL (Tool Call ID: {msg.tool_call_id}):")
            print(f"    {msg.content}")
            
        else:
            print(f"\n[{i}] ❓ ALTRO ({type(msg).__name__}):\n    {msg.content}")
            
    print("\n" + "=" * 70)

---
## 🛠️ 3. Il nostro primo Tool: Scrivere un file

Un **Tool** in LangChain è una funzione Python decorata con `@tool`.
È fondamentale aggiungere:
- **Type annotations** (tipi degli argomenti).
- Una **docstring chiara**: l'LLM legge la docstring per capire **quando** e **come** usare la funzione!

In [3]:
@tool
def scrivi_file(nome_file: str, contenuto: str) -> str:
    """
    Crea o sovrascrive un file di testo sul disco con il contenuto specificato.

    Deve essere utilizzato ogni volta che l'utente chiede:
        - di creare un file
        - di salvare dati
        - di esportare informazioni

    
    Args:
        nome_file: Il nome del file da salvare (es. 'appunti.txt', 'report.md').
        contenuto: Il testo da scrivere all'interno del file.
        
    Returns:
        Un messaggio di conferma con il numero di caratteri scritti o l'errore.
    """
    try:
        with open(nome_file, "w", encoding="utf-8") as f:
            f.write(contenuto)
        return f"File '{nome_file}' salvato con successo ({len(contenuto)} caratteri)."
    except Exception as e:
        return f"Errore durante il salvataggio del file '{nome_file}': {str(e)}"

print(f"Tool registrato: {scrivi_file.name}")
print(f"Descrizione tool: {scrivi_file.description}")
print(f"Schema argomenti: {scrivi_file.args}")

Tool registrato: scrivi_file
Descrizione tool: Crea o sovrascrive un file di testo sul disco con il contenuto specificato.

Deve essere utilizzato ogni volta che l'utente chiede:
    - di creare un file
    - di salvare dati
    - di esportare informazioni


Args:
    nome_file: Il nome del file da salvare (es. 'appunti.txt', 'report.md').
    contenuto: Il testo da scrivere all'interno del file.

Returns:
    Un messaggio di conferma con il numero di caratteri scritti o l'errore.
Schema argomenti: {'nome_file': {'title': 'Nome File', 'type': 'string'}, 'contenuto': {'title': 'Contenuto', 'type': 'string'}}


### 🔗 Binding dei Tool al Modello

Tramite il metodo `.bind_tools([tool1, tool2, ...])` forniamo ad Ollama la specifica dei tool disponibili.
Quando facciamo una richiesta che richiede un'azione, il modello non genererà testo normale, ma una richiesta strutturata di **Tool Call** contenente nome della funzione e parametri.

In [4]:
# Colleghiamo il tool al modello
llm_con_tools = llm.bind_tools([scrivi_file])

# Messaggio iniziale dell'utente
prompt_utente = "Crea un file chiamato 'benvenuto.txt' con dentro un messaggio di benvenuto al corso di AI Agentica in italiano."
messaggi: List[BaseMessage] = [HumanMessage(content=prompt_utente)]

# Prima chiamata al modello
risposta = llm_con_tools.invoke(messaggi)

print("Tipo di risposta:", type(risposta))
print("Contenuto testuale:", risposta.content or "(Nessun testo, ha generato una Tool Call!)")
print("Tool calls rilevate:", risposta.tool_calls)

Tipo di risposta: <class 'langchain_core.messages.ai.AIMessage'>
Contenuto testuale: (Nessun testo, ha generato una Tool Call!)
Tool calls rilevate: [{'name': 'scrivi_file', 'args': {'nome_file': 'benvenuto.txt', 'contenuto': 'Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali.'}, 'id': 'dccd97d6-1314-4be0-bf15-66d2b0f1da86', 'type': 'tool_call'}]


---
## 🔄 4. Anatomia del Reasoning Loop & Conversazione Multi-Turno

Osserviamo cosa succede nel ciclo completo:
1. **User Prompt** ➡️ L'utente chiede di creare un file.
2. **Tool Call** ➡️ Il modello decide di invocare `scrivi_file` con parametri `{nome_file: 'benvenuto.txt', contenuto: '...'}`.
3. **Tool Execution** ➡️ Il nostro programma Python esegue la funzione reale sul filesystem.
4. **Tool Message** ➡️ Restituiamo il risultato dell'esecuzione (Observation) al modello.
5. **Assistant Response** ➡️ Il modello conferma all'utente l'avvenuta scrittura del file.
6. **User Reply (Follow-up)** ➡️ L'utente invia un nuovo messaggio nella stessa conversazione (es. ringrazia e fa una nuova domanda).
7. **Assistant Final Reply** ➡️ L'assistente risponde tenendo conto di **tutto lo storico precedente**.

In [5]:
# Dizionario per mappare il nome del tool alla funzione Python
tools_map = {"scrivi_file": scrivi_file}

# 1. Aggiungiamo la risposta del modello (con la tool call) alla cronologia
messaggi.append(risposta)

# 2. Se il modello ha richiesto di eseguire un tool
if risposta.tool_calls:
    for tool_call in risposta.tool_calls:
        nome_tool = tool_call["name"]
        argomenti = tool_call["args"]
        call_id = tool_call.get("id", "call_1")
        
        print(f"⚙️ Esecuzione tool Python '{nome_tool}' con argomenti:")
        print(json.dumps(argomenti, indent=2, ensure_ascii=False))
        
        # Esecuzione effettiva della funzione Python
        funzione_selezionata = tools_map[nome_tool]
        risultato_tool = funzione_selezionata.invoke(argomenti)
        
        print(f"📥 Risultato ottenuto dal tool: {risultato_tool}")
        
        # 3. Inviamo l'osservazione (ToolMessage) al modello
        messaggi.append(ToolMessage(content=str(risultato_tool), tool_call_id=call_id))
    
    # 4. Chiamata al modello per formulare la risposta dopo l'esecuzione del tool
    risposta_ai_post_tool = llm_con_tools.invoke(messaggi)
    messaggi.append(risposta_ai_post_tool)
    
    print("\n💬 Risposta dell'AI all'utente:")
    print(risposta_ai_post_tool.content)
else:
    print("Il modello ha risposto direttamente:", risposta.content)

⚙️ Esecuzione tool Python 'scrivi_file' con argomenti:
{
  "nome_file": "benvenuto.txt",
  "contenuto": "Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali."
}
📥 Risultato ottenuto dal tool: File 'benvenuto.txt' salvato con successo (120 caratteri).

💬 Risposta dell'AI all'utente:
Il file 'benvenuto.txt' è stato creato con successo e contiene il seguente messaggio: "Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali.", composto da 120 caratteri.


### 👤 Nuova risposta dell'utente nella stessa chat
Simuliamo l'utente che continua la conversazione chiedendo ad esempio un chiarimento su quanto appena svolto.

In [6]:
# L'utente risponde alla conferma dell'AI
risposta_utente_2 = "Ottimo lavoro! Puoi dirmi quante parole contiene approssimativamente il messaggio che hai salvato nel file?"
print(f"👤 Nuovo messaggio dell'utente: {risposta_utente_2}")

# Aggiungiamo il nuovo messaggio utente allo storico
messaggi.append(HumanMessage(content=risposta_utente_2))

# Invochiamo il modello con l'intera cronologia
risposta_finale = llm_con_tools.invoke(messaggi)
messaggi.append(risposta_finale)

print(f"\n🤖 Risposta finale dell'assistente:\n{risposta_finale.content}")

👤 Nuovo messaggio dell'utente: Ottimo lavoro! Puoi dirmi quante parole contiene approssimativamente il messaggio che hai salvato nel file?

🤖 Risposta finale dell'assistente:
Il messaggio salvato nel file 'benvenuto.txt' contiene approssimativamente 24 parole.


### 🔍 Ispezione Visiva dell'Intera Conversazione
Ora possiamo visualizzare chiaramente ogni singolo step dello scambio (Utente $\rightarrow$ Tool Call $\rightarrow$ Tool Execution $\rightarrow$ Risposta AI $\rightarrow$ Feedback Utente $\rightarrow$ Risposta Finale).

In [7]:
# Visualizziamo la cronologia completa
mostra_conversazione(messaggi)

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] 👤 UTENTE:
    Crea un file chiamato 'benvenuto.txt' con dentro un messaggio di benvenuto al corso di AI Agentica in italiano.

[2] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'scrivi_file' | Parametri: {"nome_file": "benvenuto.txt", "contenuto": "Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali."}

[3] 📥 RISULTATO TOOL (Tool Call ID: dccd97d6-1314-4be0-bf15-66d2b0f1da86):
    File 'benvenuto.txt' salvato con successo (120 caratteri).

[4] 🤖 ASSISTENTE (AI):
    Il file 'benvenuto.txt' è stato creato con successo e contiene il seguente messaggio: "Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali.", composto da 120 caratteri.

[5] 👤 UTENTE:
    Ottimo lavoro! Puoi dirmi quante parole contiene approssimativamente il messaggio che hai salvato nel file?

[6] 🤖 ASSISTENTE (AI):
    Il messaggio salvato

In [8]:
# Verifichiamo se il file è stato effettivamente creato sul disco!
nome_file_creato = "benvenuto.txt"
if os.path.exists(nome_file_creato):
    print(f"✅ Il file '{nome_file_creato}' esiste sul disco!")
    with open(nome_file_creato, "r", encoding="utf-8") as f:
        print("--- Contenuto del file creato dall'agente ---")
        print(f.read())
        print("--------------------------------------------")
else:
    print(f"⚠️ File {nome_file_creato} non trovato.")

✅ Il file 'benvenuto.txt' esiste sul disco!
--- Contenuto del file creato dall'agente ---
Benvenuti nel corso di AI Agentica! Speriamo che questo vi aiuti a comprendere meglio il mondo degli agenti artificiali.
--------------------------------------------


---
## 🧰 5. Agente Multi-Tool: Scrittura, Lettura e Calcolo

Un agente diventa davvero potente quando ha a disposizione diversi tool specializzati e sa coordinarli in sequenza.
Aggiungiamo:
1. `leggi_file`: per ispezionare file esistenti.
2. `calcola_espressione`: per calcolare formule matematiche in modo esatto (evitando allucinazioni aritmetiche degli LLM).

In [9]:
@tool
def leggi_file(nome_file: str) -> str:
    """
    Legge il contenuto di un file di testo dal disco.
    
    Args:
        nome_file: Il nome o percorso del file da leggere.
        
    Returns:
        Il testo contenuto nel file oppure un messaggio di errore se non trovato.
    """
    try:
        if not os.path.exists(nome_file):
            return f"Errore: Il file '{nome_file}' non esiste."
        with open(file=nome_file, mode="r", encoding="utf-8") as f:
            contenuto = f.read()
        return f"Contenuto di '{nome_file}':\n{contenuto}"
    except Exception as e:
        return f"Errore durante la lettura del file '{nome_file}': {str(e)}"

@tool
def calcola_espressione(espressione: str) -> str:
    """
    Esegue un calcolo matematico sicuro ed esatto.
    
    Args:
        espressione: L'espressione aritmetica da valutare (es. '125 * 45 + 18').
        
    Returns:
        Il risultato numerico calcolato.
    """
    try:
        # Valutazione sicura di sole espressioni matematiche di base
        risultato = eval(espressione, {"__builtins__": {}}, {"abs": abs, "round": round, "min": min, "max": max})
        return f"Risultato di '{espressione}' = {risultato}"
    except Exception as e:
        return f"Errore nel calcolo di '{espressione}': {str(e)}"

lista_tool = [scrivi_file, leggi_file, calcola_espressione]
map_tool = {t.name: t for t in lista_tool}
print(f"Tool disponibili: {list(map_tool.keys())}")

Tool disponibili: ['scrivi_file', 'leggi_file', 'calcola_espressione']


### 🤖 Implementazione di un Agent Runner con Tracciamento Cronologia

Creiamo una funzione riutilizzabile `esegui_agente(richiesta, cronologia_esistente, max_iterazioni)` che gestisce la conversazione, esegue i tool ed opzionalmente stampa l'intero flusso dei messaggi.

In [10]:
from typing import Optional


def esegui_agente(
    richiesta_utente: str, 
    cronologia: Optional[List[BaseMessage]] = None, 
    max_iterazioni: int = 6, 
    verbose: bool = True,
) -> tuple[str, List[BaseMessage]]:
    """
    Esegue un loop agentico completo mantenendo e restituendo la cronologia dei messaggi.
    
    Returns:
        Una tupla con (risposta_finale_testo, cronologia_aggiornata)
    """
    llm_con_tutti_i_tool = llm.bind_tools(lista_tool)
    
    if cronologia is None:
        system_prompt = """
Usa un solo tool per volta.
Dopo aver ricevuto l'output del tool,
decidi l'azione successiva.
Non pianificare più tool contemporaneamente.
"""

        messaggi = [
            SystemMessage(content=system_prompt)
        ]
    else:
        messaggi = list(cronologia)
        
    messaggi.append(HumanMessage(content=richiesta_utente))
    
    if verbose:
        print(f"🎯 NUOVA RICHIESTA: {richiesta_utente}\n" + "="*60)
    
    for step in range(1, max_iterazioni + 1):
        if verbose:
            print(f"\n📍 [Ciclo {step}] L'agente sta ragionando...")
        
        risposta = llm_con_tutti_i_tool.invoke(messaggi)
        messaggi.append(risposta)
        
        # Se l'agente non chiama tool, ha concluso e fornito la risposta finale
        if not risposta.tool_calls:
            if verbose:
                print("🏁 L'agente ha completato il task!")
            return risposta.content, messaggi
        
        # Altrimenti eseguiamo tutti i tool richiesti
        tc = risposta.tool_calls[0]
        t_name = tc["name"]
        t_args = tc["args"]
        t_id = tc.get("id", f"call_{step}")
        
        if verbose:
            print(f"  ⚡ Azione: Invocazione Tool '{t_name}' con parametri {t_args}")
        
        if t_name in map_tool:
            risultato = map_tool[t_name].invoke(t_args)
        else:
            risultato = f"Errore: Tool '{t_name}' non trovato."
        
        if verbose:
            print(f"  👁️ Osservazione: {risultato}")
        
        messaggi.append(ToolMessage(content=str(risultato), tool_call_id=t_id))
            
    return "Limite massimo di iterazioni raggiunto senza risposta finale.", messaggi

---
## 🧪 6. Test ed Esempi Pratici di Task Autonomi

Vediamo l'agente in azione su problemi che richiedono **più passaggi coordinati**.

### Esempio A: Calcolo + Scrittura Report
L'agente deve calcolare il totale di un budget e salvare il riepilogo in un file `budget_mensile.txt`.

In [11]:
# Se il modello non riesce a distinguere le due function call consecutive prova con un altro modello

# MODEL_NAME = "qwen2.5"  # Modello raccomandato per function calling locale (o 'qwen2.5')

# llm = ChatOllama(
#     model=MODEL_NAME,
#     base_url=BASE_URL,
#     temperature=0,
# )

richiesta_1 = (
    "Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). "
    "Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato."
)

risposta_1, chat_history = esegui_agente(richiesta_1)
print("\n💬 Risultato finale:")
print(risposta_1)

🎯 NUOVA RICHIESTA: Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato.

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Azione: Invocazione Tool 'calcola_espressione' con parametri {'espressione': '850 + 320 + 145 + 75'}
  👁️ Osservazione: Risultato di '850 + 320 + 145 + 75' = 1390

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Azione: Invocazione Tool 'scrivi_file' con parametri {'contenuto': 'Affitto: 850€\nSpesa: 320€\nUtenze: 145€\nTrasporti: 75€\nTotale: 1390€', 'nome_file': 'budget_mensile.txt'}
  👁️ Osservazione: File 'budget_mensile.txt' salvato con successo (67 caratteri).

📍 [Ciclo 3] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risultato finale:
Ho creato il file 'budget_mensile.txt' con il seguente contenuto:

- Affitto: 850€
- Spesa: 320€
- Utenze: 145€
- Trasporti: 75€
- Totale: 1390€

Il file è stato salvato con

In [12]:
# Mostriamo la cronologia generata in questo primo task
mostra_conversazione(chat_history)

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    
Usa un solo tool per volta.
Dopo aver ricevuto l'output del tool,
decidi l'azione successiva.
Non pianificare più tool contemporaneamente.


[2] 👤 UTENTE:
    Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato.

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'calcola_espressione' | Parametri: {"espressione": "850 + 320 + 145 + 75"}

[4] 📥 RISULTATO TOOL (Tool Call ID: 666dd286-19e0-46b9-8aa9-635db6906b79):
    Risultato di '850 + 320 + 145 + 75' = 1390

[5] 🤖 ASSISTENTE (AI):
    Il totale delle spese mensili è di 1390€. Ora creerò il file 'budget_mensile.txt' che conterrà il dettaglio delle voci e il totale calcolato.

    🛠️ [Tool Call] Funzione: 'scrivi_file' | Parametri: {"contenuto": "Affitto: 850€\nSpesa: 320€\nUtenze: 145€\nTrasporti: 75€\nTotale: 1390€", "nome_fi

In [13]:
chat_history

[SystemMessage(content="\nUsa un solo tool per volta.\nDopo aver ricevuto l'output del tool,\ndecidi l'azione successiva.\nNon pianificare più tool contemporaneamente.\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato.", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest  ', 'created_at': '2026-10-01T12:40:22.4367599Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10613326900, 'load_duration': 6524643100, 'prompt_eval_count': 569, 'prompt_eval_duration': 1455652000, 'eval_count': 41, 'eval_duration': 2626192000, 'logprobs': None, 'model_name': 'qwen2.5:latest  ', 'model_provider': 'ollama'}, id='lc_run--01a0f77a-b72f-71b2-9dc9-cc4b9664d53a-0', tool_calls=[{'name': 'calcola_e

In [14]:
# Verifichiamo il file generato dall'agente
if os.path.exists("budget_mensile.txt"):
    with open("budget_mensile.txt", "r", encoding="utf-8") as f:
        print("Contenuto di 'budget_mensile.txt':\n")
        print(f.read())

Contenuto di 'budget_mensile.txt':

Affitto: 850€
Spesa: 320€
Utenze: 145€
Trasporti: 75€
Totale: 1390€


### Esempio B: Continuare la conversazione sullo stesso contesto
Passiamo la `chat_history` precedente al secondo turno per chiedere un'ulteriore operazione (leggere e riassumere).

In [15]:
richiesta_2 = (
    "Sulla base del file che hai appena scritto, crea un nuovo file 'riassunto_budget.txt' "
    "con un breve consiglio finanziario per risparmiare il 10%."
)

risposta_2, chat_history = esegui_agente(richiesta_2, cronologia=chat_history)
print("\n💬 Risultato finale turno 2:")
print(risposta_2)

🎯 NUOVA RICHIESTA: Sulla base del file che hai appena scritto, crea un nuovo file 'riassunto_budget.txt' con un breve consiglio finanziario per risparmiare il 10%.

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Azione: Invocazione Tool 'scrivi_file' con parametri {'nome_file': 'riassunto_budget.txt', 'contenuto': "Per risparmiare il 10% sulle tue spese mensili, potresti considerare di ridurre ciascuna delle voci di spesa del 10%. Ad esempio, l'affitto potrebbe essere ridotto di 85€, la spesa di 32€, le utenze di 14,5€ e i trasporti di 7,5€. In totale, questo ti permetterebbe di risparmiare 139€ al mese."}
  👁️ Osservazione: File 'riassunto_budget.txt' salvato con successo (294 caratteri).

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risultato finale turno 2:
Il file 'riassunto_budget.txt' è stato creato con il seguente contenuto:

"Per risparmiare il 10% sulle tue spese mensili, potresti considerare di ridurre ciascuna delle voci di spesa del 10%. Ad esempi

In [16]:
# Mostriamo la conversazione completa dopo i 2 turni
mostra_conversazione(chat_history)

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    
Usa un solo tool per volta.
Dopo aver ricevuto l'output del tool,
decidi l'azione successiva.
Non pianificare più tool contemporaneamente.


[2] 👤 UTENTE:
    Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato.

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'calcola_espressione' | Parametri: {"espressione": "850 + 320 + 145 + 75"}

[4] 📥 RISULTATO TOOL (Tool Call ID: 666dd286-19e0-46b9-8aa9-635db6906b79):
    Risultato di '850 + 320 + 145 + 75' = 1390

[5] 🤖 ASSISTENTE (AI):
    Il totale delle spese mensili è di 1390€. Ora creerò il file 'budget_mensile.txt' che conterrà il dettaglio delle voci e il totale calcolato.

    🛠️ [Tool Call] Funzione: 'scrivi_file' | Parametri: {"contenuto": "Affitto: 850€\nSpesa: 320€\nUtenze: 145€\nTrasporti: 75€\nTotale: 1390€", "nome_fi

In [73]:
chat_history

[SystemMessage(content="\nUsa un solo tool per volta.\nDopo aver ricevuto l'output del tool,\ndecidi l'azione successiva.\nNon pianificare più tool contemporaneamente.\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Calcola la spesa totale sommando: Affitto (850€), Spesa (320€), Utenze (145€), Trasporti (75€). Poi crea un file chiamato 'budget_mensile.txt' che contenga il dettaglio delle voci e il totale calcolato.", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5', 'created_at': '2026-09-21T12:47:19.193463Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2482600583, 'load_duration': 1108404333, 'prompt_eval_count': 569, 'prompt_eval_duration': 812851000, 'eval_count': 41, 'eval_duration': 556140000, 'logprobs': None, 'model_name': 'qwen2.5', 'model_provider': 'ollama'}, id='lc_run--01a0c401-a2e5-7e33-80af-cdad8610c897-0', tool_calls=[{'name': 'calcola_espressione', 'args': {

In [ ]:
# Verifichiamo il secondo file generato
if os.path.exists("riassunto_budget.txt"):
    with open("riassunto_budget.txt", "r", encoding="utf-8") as f:
        print("Contenuto di 'riassunto_budget.txt':\n")
        print(f.read())

Contenuto di 'riassunto_budget.txt':

Per risparmiare il 10% sulle tue spese mensili, potresti considerare di ridurre ciascuna delle voci di spesa del 10%. Ad esempio, l'affitto potrebbe essere ridotto di 85€, la spesa di 32€, le utenze di 14,5€ e i trasporti di 7,5€. In totale, questo ti permetterebbe di risparmiare 139€ al mese.


: 

---
## 💡 Riepilogo & Takeaways per la spiegazione

1. **Definizione di Tool**: Un tool è una normale funzione Python esposta all'LLM tramite uno schema JSON (nome, descrizione, parametri tipizzati).
2. **Tool Calling (Function Calling)**: L'LLM non esegue direttamente il codice Python per motivi di sicurezza; restituisce una richiesta di chiamata con gli argomenti identificati.
3. **Reasoning Loop (ReAct: Reason + Act)**: 
   - L'applicazione esegue la funzione reale sul sistema (es. crea file, interroga API, calcola).
   - Il risultato viene inviato all'LLM come `ToolMessage`.
   - L'LLM valuta se il compito è concluso o se sono necessari ulteriori tool.
4. **Memoria e Conversazione Multi-Turno**: Conservando la lista di messaggi (`HumanMessage`, `AIMessage`, `ToolMessage`), l'agente mantiene la consapevolezza delle azioni già compiute e dei messaggi successivi dell'utente.